# Train a proprioception-tuned CNN

We create a sensor processing model using CNN-based visual encoding finetuned with proprioception.

We create an encoding for the robot starting from a pretrained CNN model. As the feature vector of this is still large (eg 512 * 7 * 7), we reduce this to the encoding with an MLP. 

We finetune the encoding with information from proprioception.  

The sensor processing object associated with the network trained like this is in sensorprocessing/sp_propriotuned_cnn.py

In [ ]:
import sys
sys.path.append("..")

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import sensorprocessing.helper_training as helper_training
import sensorprocessing.helper_training_data as helper_training_data
from sensorprocessing.sp_propriotuned_cnn import VGG19ProprioTunedRegression, ResNetProprioTunedRegression

In [ ]:
#
# Code for deterministic run, from Robi Konievic
#
superpower=777
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
torch.use_deterministic_algorithms(True)
torch.manual_seed(superpower)
import random
random.seed(superpower)
import numpy as np
np.random.seed(superpower)
torch.backends.cudnn.benchmark = False
# torch.backends.cudnn.deterministic = True
torch.cuda.manual_seed_all(superpower)

### Exp-run initialization
Create the exp/run-s that describe the parameters of the training. 
Some of the code here is structured in such a way as to make the notebook automatizable with papermill.

In [ ]:
# *** Initialize the variables with default values
# *** This cell should be tagged as parameters
# *** If papermill is used, some of the values will be overwritten

# If it is set to discard-old, the exprun will be recreated from scratch
# creation_style = "exist-ok"
creation_style = "discard-old"

experiment = "sensorprocessing_propriotuned_cnn"
run = "vgg19_128"
# run = "resnet50_128"
# run = "vgg19_256"
# run = "resnet50_256"
# run = "boo"
# If not None, set the epochs to something different than the exp
epochs = None

# If not None, set an external experiment path
external_path = None
# If not None, set an output path
data_path = None

# Dr. Boloni's path
#external_path = pathlib.Path(Config()["experiment_external"])
# Sahara's path
# external_path = pathlib.Path("/home/sa641631/SaharaBerryPickerData/experiment_data")

In [ ]:
# create the necessary exp/run objects

if external_path:
    external_path = pathlib.Path(external_path)
    assert external_path.exists()
    Config().set_exprun_path(external_path)
    Config().copy_experiment("sensorprocessing_propriotuned_cnn")
    Config().copy_experiment("robot_al5d")
    Config().copy_experiment("demonstration")
if data_path:
    data_path = pathlib.Path(data_path)
    assert data_path.exists()
    Config().set_results_path(data_path)

# This is an example of how to run an exprun variant
# Config().create_exprun_variant("sensorprocessing_propriotuned_cnn","resnet50_128", {"epochs": 17}, new_run_name="boo")

# The experiment/run we are going to run: the specified model will be created
exp = Config().get_experiment(experiment, run, creation_style=creation_style)
exp_robot = Config().get_experiment(exp["robot_exp"], exp["robot_run"])


### Create a model that performs proprioception regression

In [ ]:
def model_training_step(model, optimizer):
    """Run one CNN training epoch and return its average loss."""
    model.train()
    total_loss = 0
    device = Config().runtime["device"]

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        loss = criterion(model(batch_X), batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(train_loader)


def model_eval_step(model):
    """Run one CNN validation epoch and return its average loss."""
    model.eval()
    total_loss = 0
    device = Config().runtime["device"]

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            total_loss += criterion(model(batch_X), batch_y).item()

    return total_loss / len(test_loader)

In [ ]:
if exp["model"] == "VGG19ProprioTunedRegression":
    model = VGG19ProprioTunedRegression(exp)
elif exp["model"] == "ResNetProprioTunedRegression":
    model = ResNetProprioTunedRegression(exp)
else:
    raise ValueError(f"Unknown model {exp['model']}")

if exp["loss"] == "MSELoss":
    criterion = nn.MSELoss()
elif exp["loss"] == "L1Loss":
    criterion = nn.L1Loss()
else:
    raise ValueError(f"Unknown loss {exp['loss']}")

device = Config().runtime["device"]
model = model.to(device)
criterion = criterion.to(device)
optimizer = optim.Adam(model.parameters(), lr=exp["learning_rate"])

will_load_existing = (
    helper_training.model_available(exp)
    and exp.get("reload_existing_model", True)
)

if not will_load_existing:
    tr = helper_training_data.load_images_as_proprioception_training(
        exp,
        exp_robot,
        generator=torch.Generator().manual_seed(superpower),
    )
    inputs_training = tr["inputs_training"]
    targets_training = tr["targets_training"]
    inputs_validation = tr["inputs_validation"]
    targets_validation = tr["targets_validation"]

    batch_size = exp["batch_size"]
    train_dataset = TensorDataset(inputs_training, targets_training)
    test_dataset = TensorDataset(inputs_validation, targets_validation)
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False
    )

model = helper_training.load_or_train(
    exp,
    model,
    optimizer,
    model_training_step,
    model_eval_step,
    epochs=epochs,
)